For every City of San Diego liability claim, find an associated Get It Done service request of the same hazard, filed before the incident and closed or still open after the incident occurred.

Association is defined as such:
1. The claim is within San Diego City. 
2. It contains relevant descriptions that map to available Get It Done services.
3. A GID service request lies within `MAX_RADIUS_M` (75 m) of the claim coordinates.
4. No work had been done when the incident happened.

In [1]:
# imports
import re
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

In [2]:
# parameters
CLAIMS_FILEPATH = 'data/claims_claims.csv'
# GID_SRC = 'https://seshat.datasd.org/get_it_done_reports'
GID_SRC = 'data'
YEARS = np.arange(2016, 2026+1)
DATA_AS_OF = '2026-11-18'

MAX_RADIUS_M = 15.0
M_TO_FT = 3.28084
EARTH_RADIUS_M = 6371000.0

Load and Basic Checks

- `asserts` confirm that each code maps to exactly one description.
- Claimant rows before the incident are dropped.
- A claim can have several claimants. Incident date, coordinates, and `org1_code` cannot disagree, but loss code can.

In [3]:
cols_to_keep = [
    'ADDRESS1','CITY','CLAIMANT_ADD_DATE','CLAIMANT_ID','CLAIM_ID','CLAIM_NUMBER','CLAIM_STATUS','INCIDENT_DATE',
    'INCIDENT_DESC','INCURRED_TOTAL','LATITUDE','LONGITUDE','LOSS_CODE','LOSS_CODE_DESC','ORG1_CODE','ORG1_DESC',
    'ORG2_CODE','ORG2_DESC','ZIP_CODE'
    ]
cols = {c: c.lower() for c in cols_to_keep}
claims = pd.read_csv(CLAIMS_FILEPATH, usecols=list(cols)).rename(columns=cols)
claims = claims[claims['incident_desc'].notna()]
claims['incident_date'] = pd.to_datetime(claims['incident_date'])
claims['claimant_add_date'] = pd.to_datetime(claims['claimant_add_date'])
claims['city'] = claims['city'].str.strip().str.title()

assert (claims.groupby('org1_code')['org1_desc'].nunique() == 1).all()
assert (claims.groupby('org2_code')['org2_desc'].nunique() == 1).all()
assert (claims.groupby('loss_code')['loss_code_desc'].nunique() == 1).all()

# claimant filed before incident
claims = claims[claims['incident_date'] <= claims['claimant_add_date']]

claimant_disagreement = claims.groupby('claim_id')[['incident_date','latitude','longitude','org1_code','loss_code']].nunique(dropna=False)
print('claims whose claimants disagree on each field:\n', (claimant_disagreement > 1).sum())

claims whose claimants disagree on each field:
 incident_date      0
latitude           0
longitude          0
org1_code          0
loss_code        426
dtype: int64


Identifying claims that can be associated to a service request

- Coordinates must be inside the City of San Diego
- `hazards` maps relevant loss codes to `service_name` that could describe the same hazard, `details_include` narrow the scope based on `service_name_detail`. `Other` service category is not included because of ambiguity. (defined below)
- `NOT_SD_CITY` removes claims outside San Diego.

In [4]:
lat_sd = claims['latitude'].between(32.5, 33.3)
lng_sd = claims['longitude'].between(-117.6, -116.8)
claims = claims[lat_sd & lng_sd]
claims = claims[claims['org1_desc'] != 'Wrong Entity']

In [5]:
# GID services (service_name) that could describe the same hazard as loss code, reviewed manually.
hazards = {
    'Maint Of Road (Potholes)':{'Pothole','Parks Issue'},
    'Maint Of Road (Other)':{'Pavement Maintenance','Parks Issue'},
    'Maintenance Of Sidewalk':{'Sidewalk Repair Issue','ROW Maintenance','Parks Issue'},
    'Trip & Fall':{'Sidewalk Repair Issue','Pavement Maintenance','ROW Maintenance','Parks Issue'},
    'Slip & Fall':{'Sidewalk Repair Issue','Pavement Maintenance','ROW Maintenance','Parks Issue'},
    'A.D.A.':{'Sidewalk Repair Issue','ADA Compliance','Traffic Engineering'},
    'Bicycle':{'Bicycle-Related Issue','Pothole','Pavement Maintenance','ROW Maintenance','Traffic Engineering','Tree Maintenance','Street Sweeping'},
    'Construction In Right Of Way':{'ROW Maintenance','Right-of-Way Code Enforcement','Pavement Maintenance','Pothole'},
    'Design Of Roadway (Lighting)':{'Street Light Maintenance'},
    'Inadequate Area Lighting':{'Street Light Maintenance','Parks Issue'},
    'Design Of Roadway (Signals)':{'Traffic Signal Issue','Traffic Signal Timing'},
    'Design Of Roadway (Signs)':{'Traffic Sign Maintenance','Traffic Engineering','ROW Maintenance'},
    'Design Of Roadway (Crosswalk)':{'ROW Maintenance','Traffic Engineering'},
    'Design Of Roadway (Sight Distance)':{'Traffic Engineering','Vegetation Encroachment','Tree Maintenance'},
    'Storm Drains':{'Stormwater','Street Flooded','ROW Maintenance','Parks Issue'},
    'Storm Drain Funding':{'Stormwater','Street Flooded','ROW Maintenance'},
    'City Tree':{'Tree Maintenance','Parks Issue'},
    'Tree Roots':{'Tree Maintenance','Parks Issue'},
    'Foliage Obstruction':{'Tree Maintenance','Vegetation Encroachment','Weed Cleanup'},
    'Missing/Damaged Trash Can':{'Trash/Recycling Collection','Container Request','Missed Collection'},
    }

claims = claims[claims['loss_code_desc'].isin(list(hazards))]

NOT_SD_CITY = {
    'Escondido','Rancho Santa Fe','Poway','Santee',
    'Solana Beach','Alpine','Bonita','Fallbrook',
    'Encinitas','Imperial Beach','Boardman','San'
    }
claims = claims[~claims['city'].isin(NOT_SD_CITY)]

In [6]:
# aggregating
claims = claims.sort_values(['claim_id','incurred_total'], ascending=[True, False])
claims_agg = (claims.groupby('claim_id', as_index=False).agg(
        claimant_count=('claimant_id','count'),
        org1_desc=('org1_desc','first'),
        org2_desc=('org2_desc','first'),
        claim_number=('claim_number','first'),
        address1=('address1','first'),
        loss_code_desc=('loss_code_desc','first'),
        loss_codes_all=('loss_code_desc', lambda t: '|'.join(sorted(set(t)))),
        incident_desc=('incident_desc','first'),
        incident_date=('incident_date','first'),
        incurred_total=('incurred_total','sum'),
        claimant_add_date=('claimant_add_date','min'),
        latitude=('latitude','first'),
        longitude=('longitude','first')
    ))
claims_agg['addr_has_number'] = claims_agg['address1'].astype(str).str.match(r'^\s*\d')
print(len(claims_agg), 'unique claims')

11970 unique claims


Load and Basic Cleaning

In [7]:
pub_cols = [
    'service_request_id','service_request_parent_id','date_requested','date_closed','status',
    'case_record_type','service_name','service_name_detail','lat', 'lng','public_description'
    ]
closed = pd.concat([pd.read_csv(f'{GID_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=pub_cols, low_memory=False) for y in YEARS], ignore_index=True)
open_ = pd.read_csv(f'{GID_SRC}/get_it_done_requests_open_datasd.csv', usecols=pub_cols)

public = pd.concat([closed, open_], ignore_index=True).drop_duplicates('service_request_id')
public = public[public['service_request_parent_id'].isna()].drop(columns='service_request_parent_id')
public['date_requested'] = pd.to_datetime(public['date_requested']).dt.normalize()
public['date_closed'] = pd.to_datetime(public['date_closed']).dt.normalize()
public = public.dropna(subset=['lat', 'lng', 'service_name'])
public = public[(public['lat'] != 0) & (public['lng'] != 0)].reset_index(drop=True)

In [8]:
# looks at what services are not present in hazards
all_services = public['service_name'].unique()
named_in_hazards = set().union(*hazards.values())
print('GID services not in hazards:')
print(public.loc[~public['service_name'].isin(named_in_hazards), 'service_name'].value_counts())
print('in hazards but not in data:', sorted(named_in_hazards - set(all_services)))

GID services not in hazards:
service_name
Illegal Dumping                            473702
Parking                                    262117
Encampment                                 230188
Parking Violation                          176475
Graffiti                                   155750
Graffiti - Public                          118811
Other                                      108306
Shared Mobility Device                      63586
Graffiti - Code Enforcement                 59482
Parking - 72-Hours                          50346
Environmental Services Code Compliance      46113
Dead Animal                                 33958
Stormwater Code Enforcement                 28838
Waste on Private Property                   16158
Graffiti - Private                          12825
Development Services - Code Enforcement      9856
Oversized Vehicle                            9073
ESD Collections                              7729
STRO Violation                               4258
Building

Matching `claims` to `service_requests`

Every request within `MAX_RADIUS_M` of a claim is a candidate.

- Only claims whose incident occurred after GID pilot are eligible.
- Only requests filed before the incident are kept.

In [9]:
first_req, last_req = public['date_requested'].min(), public['date_requested'].max()
gid_range = claims_agg['incident_date'].between(first_req, last_req)
cand_claims = claims_agg[gid_range].reset_index(drop=True)

tree = BallTree(np.radians(public[['lat','lng']].to_numpy()), metric='haversine')
ind, dist = tree.query_radius(np.radians(cand_claims[['latitude','longitude']].to_numpy()),
                              r=MAX_RADIUS_M / EARTH_RADIUS_M, return_distance=True)
n_per_claim = [len(i) for i in ind]
pairs = pd.DataFrame({
    'ci': np.repeat(np.arange(len(cand_claims)), n_per_claim),
    'gi': np.concatenate(ind) if sum(n_per_claim) else np.array([], dtype=int),
    'distance_ft': (np.concatenate(dist) * EARTH_RADIUS_M * M_TO_FT) if sum(n_per_claim) else np.array([])})
cand = (pairs.join(cand_claims.add_suffix('_claim'), on='ci')
             .join(public.add_suffix('_gid'), on='gi'))
cand = cand.rename(columns={
    'claim_id_claim':'claim_id', 'incident_date_claim':'incident_date',
    'date_requested_gid':'date_requested', 'service_name_gid':'service_name',
    'loss_code_desc_claim':'loss_code_desc', 'incurred_total_claim':'incurred_total'
    })
cand['lead_days'] = (cand['incident_date'] - cand['date_requested']).dt.days
cand = cand[cand['lead_days'] >= 0].reset_index(drop=True)
print(len(cand), 'candidate pairs for', cand['claim_id'].nunique(), 'claims')

66828 candidate pairs for 5827 claims


Checks and Associations

- `topic_ok` the request's service is in `hazard[loss code]` and fits `details`
- `reports_damage` the request text mentions damage, injury, etc.
- `request_status` relative to the incident date
    - `closed_after_incident`
    - `closed_same_day`
    - `closed_before_incident`
    - `referred`
    - `open`
    - `unknown`

`associated` = topic fits, state is `closed_after_incident` or `open`, filed at least a day before the incident

In [10]:
# narrows the scope based on service_name_detail using re
details_include = {
    ('Bicycle','ROW Maintenance'):r'bic|bike',
    ('Bicycle','Traffic Engineering'):r'bic|bike',
    ('Bicycle','Pavement Maintenance'):r'bike|asphalt|pavement',
    ('Bicycle','Tree Maintenance'):r'bike',
    ('Maintenance Of Sidewalk','ROW Maintenance'):r'(?<!paint )curb|bridge|stairs|railing',
    ('Trip & Fall','ROW Maintenance'):r'(?<!paint )curb|bridge|stairs|railing|mud debris',
    ('Slip & Fall','ROW Maintenance'):r'(?<!paint )curb|bridge|stairs|railing|mud debris',
    ('Construction In Right Of Way','ROW Maintenance'):r'trench|mud debris',
    ('Design Of Roadway (Signs)','ROW Maintenance'):r'\bsign\b|signage',
    ('Design Of Roadway (Signs)','Traffic Engineering'):r'\bsign\b|signage',
    ('Design Of Roadway (Crosswalk)','ROW Maintenance'):r'striping|legend|markers',
    ('Design Of Roadway (Crosswalk)','Traffic Engineering'):r'marking|striping|curb ramp',
    ('Design Of Roadway (Sight Distance)','Traffic Engineering'):r'visibility',
    ('A.D.A.','Traffic Engineering'):r'\bADA\b|curb ramp',
    ('Storm Drains','ROW Maintenance'):r'flood|mud debris|ponding',
    ('Storm Drain Funding','ROW Maintenance'):r'flood|mud debris|ponding',
    ('Storm Drains','Parks Issue'):r'storm drain',
    ('Maint Of Road (Potholes)','Parks Issue'):r'pothole|pavement(?! striping)|walkway',
    ('Maint Of Road (Other)','Parks Issue'):r'pothole|pavement|walkway|pathway',
    ('Maintenance Of Sidewalk','Parks Issue'):r'sidewalk/|pathway|walkway',
    ('Trip & Fall','Parks Issue'):r'pothole|pavement(?! striping)|walkway|pathway|sidewalk/|trail',
    ('Slip & Fall','Parks Issue'):r'pothole|pavement(?! striping)|walkway|pathway|sidewalk/|trail',
    ('Inadequate Area Lighting','Parks Issue'):r'lighting|electrical',
    ('City Tree','Parks Issue'):r'tree',
    ('Tree Roots','Parks Issue'):r'tree',
    ('Missing/Damaged Trash Can','Trash/Recycling Collection'):r'damaged container|stolen',
}

# details that are non hazards
details_exclude = (
    r'PLAN CHECK|GRANTS|COST ESTIMATE|PUBLIC RECORDS|DATA COLLECTION|FINANCE PLAN|FEASIBILITY|REQUEST FOR INFORMATION|INFESTATION'
    r'|FREE TREE PLANTING|STUMP GRIND|COST SHARE|FEEDBACK|UNFUNDED NEEDS|GENERAL INFORMATION|CITIZEN CONTACT|EVALUATE FOR RACKS')

def topic_match(loss, service, detail):
    if service not in hazards.get(loss, ()):
        return False
    if pd.isna(detail) or not str(detail).strip():
        if (loss, service) in details_include:
            return False
    detail = str(detail).strip()
    if re.search(details_exclude, detail, flags=re.IGNORECASE):
        return False
    d = details_include.get((loss, service))
    if d is None:
        return True
    return bool(re.search(d, detail, flags=re.IGNORECASE))

cand['topic_ok'] = [topic_match(loss, service, detail) for loss, service, detail in zip(cand['loss_code_desc'], cand['service_name'], cand['service_name_detail_gid'])]

In [11]:
DAMAGE_RE = (
    r'\bflat tire\b|\bblew (?:a |my )?tire\b|\bblown tire\b|\btires?\b|\brims?\b|\bwheel (?:bent|damage)\b|\balignment\b'
    r'|\bmy (?:car|vehicle|truck|bike|bicycle)\b|\bdamage to (?:my|our)\b|\bdamaged my\b'
    r'|\binjur(?:y|ed|ies)\b|\bi (?:fell|tripped|slipped|crashed)\b|\bfell (?:and|on|in|into|over)\b|\bhit (?:a|the|this) (?:pothole|hole|bump|curb)\b'
    r'|\baccident\b|\bcrash(?:ed)?\b|\bhospital\b|\bambulance\b')
assert re.search(DAMAGE_RE, 'got a flat tire here', re.IGNORECASE) and not re.search(DAMAGE_RE, 'needs a tree trim, entire block', re.IGNORECASE)
cand['reports_damage'] = cand['public_description_gid'].fillna('').str.contains(DAMAGE_RE, case=False, regex=True)

# state of each request relative to the incident date (all arguments are aligned pandas Series)
def request_status(status, date_requested, date_closed, incident_date):
    valid_close = date_closed.notna() & (date_closed >= date_requested)
    closed = status.eq('Closed')
    return np.select(
        [status.eq('Referred'),
         closed & valid_close & (date_closed > incident_date),
         closed & valid_close & (date_closed == incident_date),
         closed & valid_close,
         ~closed & date_closed.isna()],
        ['referred', 'closed_after_incident', 'closed_same_day', 'closed_before_incident', 'open'], default='unknown')

# the two statuses that mean the City was notified but had not done the work when the incident happened
WORK_AFTER_INCIDENT = ['closed_after_incident', 'open']

cand['request_status'] = request_status(cand['status_gid'], cand['date_requested'], cand['date_closed_gid'], cand['incident_date'])
cand['associated'] = (
    cand['topic_ok'] 
    & cand['request_status'].isin(WORK_AFTER_INCIDENT) 
    & (cand['lead_days'] >= 1) 
    )
print(cand['request_status'].value_counts().to_dict(), '| associated pairs:', int(cand['associated'].sum()))
print('associated claims whose request mentions prior vehicle damage or injury:',
      int(cand[cand['associated']].groupby('claim_id')['reports_damage'].max().sum()))

{'closed_before_incident': 54502, 'referred': 9050, 'closed_after_incident': 2804, 'open': 366, 'closed_same_day': 94, 'unknown': 12} | associated pairs: 660
associated claims whose request mentions prior vehicle damage or injury: 76


Timeline

In [12]:
DATA_AS_OF = pd.Timestamp(DATA_AS_OF) if DATA_AS_OF else max(public['date_requested'].max(), public['date_closed'].max())
print('open requests are measured up to', DATA_AS_OF.date())

cand['work_timing'] = np.where(
    cand['associated'],
    np.where(cand['request_status'] == 'open', 'still_open', 'closed_after_incident'), 
    'not_associated'
    )
end = cand['date_closed_gid'].fillna(DATA_AS_OF)
cand['days_unresolved_after_incident'] = (end - cand['incident_date']).dt.days
cand['total_days_open'] = (end - cand['date_requested']).dt.days
cand['days_incident_to_close'] = (cand['date_closed_gid'] - cand['incident_date']).dt.days
cand['days_notice_to_close'] = (cand['date_closed_gid'] - cand['date_requested']).dt.days
cand['closure_vs_claim_filed'] = np.where(
    cand['associated'],
    np.select([cand['date_closed_gid'].isna(), cand['date_closed_gid'] <= cand['claimant_add_date_claim']], ['still_open', 'closed_before_claim_filed'], default='closed_after_claim_filed'), 
    'not_associated'
    )

open requests are measured up to 2026-11-18


For every claim, keep the best candidate. An associated case if any exists, then the closest, then the most recent relative to the incident. `n_candidates` records how many candidates the claim had (a dense area makes coincidence more likely), and `earliest_associated_request` records the earliest associated request (longest notice).

In [13]:
cand = cand.sort_values(['claim_id','associated','distance_ft','date_requested'], ascending=[True, False, True, False])
n_cand = cand.groupby('claim_id')['claim_id'].transform('size')
first_assoc = cand[cand['associated']].groupby('claim_id')['date_requested'].min().rename('earliest_associated_request')
liabilities = (cand.assign(n_candidates=n_cand).drop_duplicates('claim_id')
                   .join(first_assoc, on='claim_id').reset_index(drop=True))
assoc = liabilities[liabilities['associated']]

In [14]:
assoc[[
    'claim_id','loss_code_desc','service_request_id_gid','service_name','service_name_detail_gid','distance_ft','lead_days',
    'request_status','work_timing','days_unresolved_after_incident','total_days_open','days_incident_to_close',
    'days_notice_to_close','closure_vs_claim_filed','n_candidates','earliest_associated_request','incident_date',
    'date_requested','date_closed_gid','incurred_total','incident_desc_claim','public_description_gid'
     ]].to_csv(f'associated_claims_{MAX_RADIUS_M}_m.csv', index=False)

cand_claims[['claim_id', 'loss_code_desc', 'incident_date', 'incurred_total']].to_csv('eligible_claims.csv', index=False)